# FlakeForge RL Training

### Reinforcement Learning Agent for Autonomous Flaky Test Diagnosis and Fix



Architecture: Unified Agent + GRPO (Group Relative Policy Optimization)  

Reward Type: Verifiable multi-signal reward (no LLM judge)  

Environment: FlakeForgeEnvironment with local pytest execution  



---



## Table of Contents

1. [Phase 0: Prerequisites & Environment Setup](#phase0)

2. [Phase 1: Project Structure](#phase1)

3. [Phase 2: Model Setup (Qwen + LoRA)](#phase2)

4. [Phase 3: Environment + Runner Compatibility](#phase3)

5. [Phase 4: Reward Function Compatibility](#phase4)

6. [Phase 5: Curriculum Learning](#phase5)

7. [Phase 6: GRPO Config](#phase6)

8. [Phase 7: Trainer Wiring](#phase7)

9. [Phase 8: Anti-Reward-Hacking Checks](#phase8)

10. [Phase 9: Launch Commands](#phase9)


---
<a id='phase0'></a>
## Phase 0: Prerequisites & Environment Setup

### Hardware Requirements
- **Minimum**: 2x A100 80GB (or 4x A6000 48GB)
- **Recommended**: 4x A100 80GB for stable GRPO with 7B model
- **RAM**: Large system RAM helps when many curriculum repos are open and many pytest processes run in parallel
- **Storage**: 2TB NVMe SSD (100 repos × clones × rollouts = massive I/O)

In [ ]:
# Phase 0 setup for THIS repository (FlakeForge)
%pip install -q -r server/requirements.txt
%pip install -q trl transformers datasets peft accelerate wandb python-dotenv

from pathlib import Path
import os
from dotenv import load_dotenv
load_dotenv()
print('Workspace:', Path.cwd())
print('FF_REPO_PATH =', os.getenv('FF_REPO_PATH', 'test_repos/timing_race_minimal'))
print('FF_TEST_ID =', os.getenv('FF_TEST_ID', 'tests/test_flaky.py::test_fetch_should_complete'))


---
<a id='phase1'></a>
## Phase 1: Project Structure

```
flakeforge/
├── training/
│   ├── train_grpo.py           # Main training entry point
│   ├── grpo_trainer.py         # Custom GRPO trainer
│   ├── reward_model.py         # Manifest-grounded reward fn
│   └── curriculum.py           # Curriculum scheduler
├── environment/
│   ├── docker_env.py           # Docker environment manager
│   ├── episode_runner.py       # Single episode orchestrator
│   ├── tool_executor.py        # fetch_logs, CHAOS_PROBE, etc.
│   └── patch_applier.py        # Unified diff application
├── model/
│   ├── model_loader.py         # Qwen-2.5-7B + LoRA setup
│   ├── tokenizer_utils.py      # Special token handling
│   └── generation_config.py    # Sampling parameters
├── data/
│   ├── manifests/              # flake_manifest.json files
│   ├── repositories/           # Git repos with flaky tests
│   └── curriculum_stages/      # Stage 1/2/3 dataset splits
├── configs/
│   ├── grpo_config.yaml        # All hyperparameters
│   └── deepspeed_z3.json       # DeepSpeed ZeRO-3 config
└── evaluation/
    ├── eval_runner.py           # Held-out evaluation suite
    └── metrics.py               # FRR, precision, reward curves
```

In [ ]:
# Validate current FlakeForge project structure (no shadow project creation)

from pathlib import Path
required_paths = [
    Path('models.py'),
    Path('agent/unified_agent.py'),
    Path('server/FlakeForge_environment.py'),
    Path('server/reward.py'),
    Path('training/grpo_trainer.py'),
    Path('inference.py'),
    Path('test_repos'),
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing required project paths: {missing}')
print('Project layout check passed for FlakeForge.')

---
<a id='phase2'></a>
## Phase 2: Model Setup (Qwen-2.5-7B + LoRA)

We use LoRA (Low-Rank Adaptation) to make 7B training feasible on limited GPU memory.
- **LoRA rank r=64** gives strong capacity while keeping trainable params ~1.1% of total.
- **Flash Attention 2** gives 2× memory efficiency.
- **bfloat16** is more numerically stable than float16 for RL training.

In [ ]:
# Compatible model/tokenizer loader for current FlakeForge + GRPO stack

from __future__ import annotations
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model
def load_model_and_tokenizer(model_name: str | None = None):
    model_name = model_name or os.getenv('MODEL_NAME') or 'Qwen/Qwen2.5-Coder-7B-Instruct'

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        padding_side='left',
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map='auto'
    )
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=32,
        lora_alpha=64,
        lora_dropout=0.05,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        bias='none',
        inference_mode=False,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model, tokenizer
print('load_model_and_tokenizer() is ready for this environment.')


---
<a id='phase3'></a>
## Phase 3: Environment + test runner

Each training episode uses `FlakeForgeEnvironment` with **local pytest** subprocesses on the seed repo:
- **Isolated `rootdir` / `confcutdir`** — pytest does not import the parent FlakeForge tree as the test package
- **Multiple pytest runs per step** — stable pass-rate estimates from repeated execution
- **ThreadPoolExecutor** — parallel reps when the runner supports `run_test_n_times`

In [ ]:
# Use the real FlakeForge environment and local pytest runner

from __future__ import annotations



import os

from pathlib import Path
from server.FlakeForge_environment import FlakeForgeEnvironment
from server.docker_runner import DockerTestRunner
def build_env(
    repo_path: str | None = None,
    test_identifier: str | None = None,
    max_steps: int = 8,
    num_runs: int = 10,
) -> FlakeForgeEnvironment:
    repo_path = repo_path or os.getenv('FF_REPO_PATH', 'test_repos/timing_race_minimal')
    test_identifier = test_identifier or os.getenv('FF_TEST_ID', 'tests/test_flaky.py::test_fetch_should_complete')
    runner = DockerTestRunner(repo_path)
    env = FlakeForgeEnvironment(
        repo_path=repo_path,
        test_identifier=test_identifier,
        max_steps=max_steps,
        num_runs=num_runs,
        runner=runner,
    )
    return env



def probe_preflight(repo_path: str | None = None, test_identifier: str | None = None):
    env = build_env(repo_path=repo_path, test_identifier=test_identifier, max_steps=4, num_runs=6)
    obs = env.reset(preflight_quick_runs=4, preflight_confirm_runs=6)
    print('episode_id:', obs.episode_id)
    print('env_type:', obs.env_type)
    print('should_train:', obs.should_train)
    print('baseline_pass_rate:', round(obs.baseline_pass_rate, 3))
    print('failure_frontier:', obs.failure_frontier)
    print('top_causal_hints:', obs.causal_hints[:5])
    return env, obs

print('Environment helpers ready. Call probe_preflight() for a smoke check.')


---

<a id='phase4'></a>

## Phase 4: Reward Function Compatibility



Current FlakeForge reward is deterministic and multi-signal (no LLM judge):



| Component | Range | Purpose |

|---|---|---|

| Format Reward | 0.0 to 1.0 | Valid structured think + patch format |

| Compile Reward | -1.0 to 1.0 | Patch apply/compile validity gate |

| Stability Reward | ~-1.0 to 2.0 | Pass-rate improvement shaping |

| Causal Proximity | -0.3 to 1.0 | Patching near failure frontier |

| Entropy Reward | -0.5 to 1.0 | Reducing failure-mode entropy |

| Anti-hack Penalty | -2.0 to 0.0 | Penalize reward-hacking patterns |



Hard-gate behavior: if format/compile/anti-hack fail, rollout reward short-circuits early.


In [ ]:
# Reward compatibility demo using current FlakeForge architecture

from models import (

    FlakeForgeAction,

    StructuredThink,

    ThinkClaim,

    StructuredPatch,

    PatchHunk,

)

from server.reward import (

    compute_format_reward,

    compute_compile_reward,

    compute_reasoning_consistency,

    compute_anti_hack_penalty,

)



claim = ThinkClaim(

    claim_id='c1',

    category='async_wait',

    entity='fetch_data_with_race',

    location='source.py::fetch_data_with_race',

    polarity='present',

    reason='Timeout in wait_for is too aggressive for observed latency',

)

structured_think = StructuredThink(claims=[claim], confidence=0.92, format_penalty=0.0)



hunk = PatchHunk(

    hunk_id='h1',

    file='source.py',

    search='result = await asyncio.wait_for(task, timeout=0.05)',

    replace='result = await asyncio.wait_for(task, timeout=0.50)',

    rationale='Align timeout with operation latency',

    addresses_claim='c1',

)

structured_patch = StructuredPatch(hunks=[hunk], format_penalty=0.0)



patch_text = '''--- source.py

<<<<<<< SEARCH

result = await asyncio.wait_for(task, timeout=0.05)

=======

result = await asyncio.wait_for(task, timeout=0.50)

>>>>>>> REPLACE'''



action = FlakeForgeAction(

    raw_response='demo',

    think_text='{"claims": [{"category": "async_wait"}], "confidence": 0.92}',

    patch_text=patch_text,

    structured_think=structured_think,

    structured_patch=structured_patch,

    predicted_category='async_wait',

    predicted_confidence=0.92,

)



format_reward = compute_format_reward(action)

compile_reward = compute_compile_reward(True)

consistency_reward = compute_reasoning_consistency(

    predicted_category='async_wait',

    inferred_category_from_patch='async_wait',

    think_text=action.think_text,

    patch_text=action.patch_text,

)

anti_hack = compute_anti_hack_penalty(action.patch_text, ['source.py'], lines_changed=1)



print('format_reward:', format_reward)

print('compile_reward:', compile_reward)

print('reasoning_consistency:', consistency_reward)

print('anti_hack_penalty:', anti_hack)


---
<a id='phase5'></a>
## Phase 5: Curriculum Learning

The model must learn incrementally — throwing hard async flakes at it from episode 1 **destroys** the training signal.

| Stage | Episodes | Flake Types | Advance Threshold |
|---|---|---|---|
| Stage 1 | 0–30 | TIMING_SENSITIVE, RANDOM_SEED, ORDER_DEPENDENT | Mean reward > 3.0 |
| Stage 2 | 31–65 | ASYNC_DEADLOCK, RACE_CONDITION, MOCK_POLLUTION | Mean reward > 2.0 |
| Stage 3 | 66–100 | FILESYSTEM_RACE, NETWORK_TIMEOUT, DATABASE_STATE | Mean reward > 1.5 |

In [ ]:
# Curriculum helper aligned to existing seed repos

from __future__ import annotations



import random

from dataclasses import dataclass

from pathlib import Path



@dataclass

class Stage:

    name: str

    repos: list[str]

    min_episodes: int

    threshold: float



class RepoCurriculum:

    def __init__(self, root: str = 'seed_repos'):

        root_path = Path(root)

        all_repos = sorted([p.name for p in root_path.iterdir() if p.is_dir()]) if root_path.exists() else []



        easy = [r for r in all_repos if r in {'nondeterminism', 'shared_state', 'order_dependency'}]

        medium = [r for r in all_repos if r in {'timing_race', 'resource_leak', 'external_dependency'}]

        hard = [r for r in all_repos if r not in set(easy + medium)]



        self.stages = [

            Stage('stage_easy', easy, 20, 1.0),

            Stage('stage_medium', medium, 30, 0.8),

            Stage('stage_hard', hard, 40, 0.6),

        ]

        self.stage_idx = 0

        self.episode_count = 0

        self.rewards: list[float] = []



    @property

    def current(self) -> Stage:

        return self.stages[self.stage_idx]



    def sample_repo(self) -> str:

        repos = self.current.repos or [s for st in self.stages for s in st.repos]

        if not repos:

            raise RuntimeError('No repositories found under seed_repos/')

        return random.choice(repos)



    def update(self, reward: float):

        self.episode_count += 1

        self.rewards.append(reward)

        if len(self.rewards) < 10:

            return

        mean_recent = sum(self.rewards[-10:]) / 10

        stage = self.current

        if (

            self.episode_count >= stage.min_episodes

            and mean_recent >= stage.threshold

            and self.stage_idx < len(self.stages) - 1

        ):

            self.stage_idx += 1

            print(f'Advanced to {self.current.name}')



curriculum = RepoCurriculum('seed_repos')

print('Current stage:', curriculum.current.name)

print('Sample repo:', curriculum.sample_repo())


---
<a id='phase6'></a>
## Phase 6: GRPO Hyperparameter Configuration

### Critical Knobs

| Parameter | Value | Why |
|---|---|---|
| `grpo_group_size` (G) | 8 | Sample 8 rollouts per repo — core of GRPO |
| `learning_rate` | 1e-6 | Very low — RL is noisy, prevent forgetting |
| `kl_coef` | 0.04 | Prevents drift from base model |
| `clip_range` | 0.2 | Standard PPO clip |
| `max_grad_norm` | 0.5 | Gradient clipping — critical for RL |
| `temperature` | 0.8 | Exploration during rollout |

### KL Coefficient Decision Tree
- Reward collapses → **increase** `kl_coef` (0.04 → 0.1)
- Reward grows too slowly → **decrease** `kl_coef` (0.04 → 0.01)
- Valid range: **0.01 – 0.2**

In [ ]:
# Notebook-level GRPO config compatible with training/grpo_trainer.py

import json

from pathlib import Path



grpo_notebook_config = {

    'model_name': 'Qwen/Qwen2.5-Coder-7B-Instruct',

    'output_dir': './outputs/flakeforge_grpo_notebook',

    'sft_data_path': None,

    'use_execution': False,

    'batch_size': 2,

    'num_epochs': 1,

    'learning_rate': 5e-6,

    'logging_steps': 10,

    'save_steps': 100,

    'max_completion_length': 2048,

    'num_generations': 4,

}



cfg_path = Path('training/grpo_notebook_config.json')

cfg_path.write_text(json.dumps(grpo_notebook_config, indent=2), encoding='utf-8')



print('Wrote compatible config to', cfg_path)

print(json.dumps(grpo_notebook_config, indent=2))


---
<a id='phase7'></a>
## Phase 7: The GRPO Training Loop

### How GRPO Works (vs PPO)
- **PPO** needs a separate Value Network to estimate baselines → memory + complexity
- **GRPO** samples G responses per prompt, normalizes rewards within the group as the advantage → **no value network needed**
- This makes GRPO ideal for code generation tasks with sparse rewards

In [ ]:
# Unified prompt/action utilities for current agent architecture

from __future__ import annotations



from typing import Any



from agent.unified_agent import (

    build_unified_prompt,

    extract_think,

    extract_patch,

    extract_category_from_think,

    extract_confidence_from_think,

)

from models import FlakeForgeAction



def action_from_completion(completion: str) -> FlakeForgeAction:

    think = extract_think(completion)

    patch = extract_patch(completion)

    return FlakeForgeAction(

        raw_response=completion,

        think_text=think,

        patch_text=patch,

        predicted_category=extract_category_from_think(think),

        predicted_confidence=extract_confidence_from_think(think),

    )



def preview_prompt(observation: Any, max_chars: int = 1200) -> str:

    prompt = build_unified_prompt(observation)

    print('Prompt length:', len(prompt))

    print(prompt[:max_chars])

    return prompt



print('Unified prompt/action utilities ready.')


In [ ]:
# GRPO trainer wiring using the actual training/grpo_trainer.py

from __future__ import annotations



import json

from pathlib import Path



from training.grpo_trainer import create_trainer, build_reward_function



def build_trainer_from_config(config_path: str = 'training/grpo_notebook_config.json'):

    cfg = json.loads(Path(config_path).read_text(encoding='utf-8'))

    trainer = create_trainer(

        model_name=cfg['model_name'],

        output_dir=cfg['output_dir'],

        sft_data_path=cfg.get('sft_data_path'),

        use_execution=cfg.get('use_execution', False),

        batch_size=cfg.get('batch_size', 2),

        num_epochs=cfg.get('num_epochs', 1),

        learning_rate=cfg.get('learning_rate', 5e-6),

        logging_steps=cfg.get('logging_steps', 10),

        save_steps=cfg.get('save_steps', 100),

        max_completion_length=cfg.get('max_completion_length', 2048),

        num_generations=cfg.get('num_generations', 4),

    )

    return trainer, cfg



def dry_run_reward_fn():

    reward_fn = build_reward_function(use_execution=False)

    prompts = ['diagnose and patch flaky test']

    completions = ['''<think>Root Cause: async_wait (confidence: 0.9)</think>\n<patch>--- source.py\n<<<<<<< SEARCH\nold\n=======\nnew\n>>>>>>> REPLACE</patch>''']

    rewards = reward_fn(prompts, completions)

    print('offline reward sample:', rewards)

    return rewards



print('Trainer helpers ready. Run build_trainer_from_config() when dependencies/model are available.')

_ = dry_run_reward_fn()


---

<a id='phase8'></a>

## Phase 8: Anti-Reward-Hacking Safeguards



| Hack Attempt | Current Defense |

|---|---|

| Inject `time.sleep` / artificial waits | `compute_anti_hack_penalty` negative shaping |

| Blanket test skipping (`@pytest.mark.skip`) | Strong anti-hack penalty |

| Bare `except:` suppression | Anti-hack penalty |

| Removing assertions to force pass | Assertion-removal penalty |

| Empty or malformed patch | Format gate + compile gate short-circuit |

| No-op patchs | No-op penalty + limited downstream reward |


In [ ]:
# Sanity checks for anti-hack and format gates in current reward system

from models import FlakeForgeAction

from server.reward import compute_anti_hack_penalty, compute_format_reward



hack_patch = '''--- tests/test_x.py

<<<<<<< SEARCH

assert result == 1

=======

import time

time.sleep(2)

>>>>>>> REPLACE'''



good_patch = '''--- source.py

<<<<<<< SEARCH

result = await asyncio.wait_for(task, timeout=0.05)

=======

result = await asyncio.wait_for(task, timeout=0.50)

>>>>>>> REPLACE'''



hack_action = FlakeForgeAction(

    raw_response='hack',

    think_text='Root Cause: async_wait (confidence: 0.5)',

    patch_text=hack_patch,

    predicted_category='async_wait',

    predicted_confidence=0.5,

)



good_action = FlakeForgeAction(

    raw_response='good',

    think_text='Root Cause: async_wait (confidence: 0.9)',

    patch_text=good_patch,

    predicted_category='async_wait',

    predicted_confidence=0.9,

)



print('hack_format_reward:', compute_format_reward(hack_action))

print('hack_anti_hack_penalty:', compute_anti_hack_penalty(hack_action.patch_text, ['tests/test_x.py'], 3))

print('good_format_reward:', compute_format_reward(good_action))

print('good_anti_hack_penalty:', compute_anti_hack_penalty(good_action.patch_text, ['source.py'], 1))


---
<a id='phase9'></a>
## Phase 9: Launch Commands

Run these in your terminal after completing all setup phases.

In [ ]:
# Launch commands aligned with this repository

print('1) Run inference smoke test (local pytest in the seed repo):')

print('   python inference.py --repo-path test_repos/timing_race_minimal --test-id tests/test_flaky.py::test_fetch_should_complete --num-runs 6')

print('')

print('2) Start OpenEnv server (if using remote env mode later):')

print('   python -m server.app --port 5000')

print('')

print('3) Start GRPO training from module code:')

print("   from training.grpo_trainer import create_trainer; trainer = create_trainer(...); trainer.train()")

print('')

print('4) Use notebook helpers: probe_preflight(), build_trainer_from_config()')


---
## 🔧 Key Stability Rules (Quick Reference)

1. **Reward collapses to 0** → Increase `kl_coef`: `0.04 → 0.1` (model is drifting)
2. **Reward grows too slowly** → Decrease `kl_coef`: `0.04 → 0.01` (over-constrained)
3. **Gradient explosions** → Decrease `max_grad_norm`: `0.5 → 0.1`
4. **`<think>` block disappears** → Increase format penalty to `-3.0`
5. **Start with G=8**, not higher — many parallel pytest subshells can bottleneck on CPU/disk, not just GPU
6. **Never skip curriculum** — hard async flakes from episode 1 destroy early training signal
7. **Check W&B `reward_std`** — if it collapses, GRPO has no gradient signal (all rollouts identical)

---
*FlakeForge RL Training Notebook — built with GRPO + Qwen-2.5-7B + Manifest-Grounded RLVR*